In [5]:
import pyreadstat

path = "../data/raw/CY09_MS_STU_PUF.sav"

df_meta, meta = pyreadstat.read_sav(path, metadataonly=True)

print("Переменных:", len(meta.column_names))
print("Наблюдений:", meta.number_rows)


Переменных: 885
Наблюдений: 755721


In [6]:
ai_vars = {
    name: label
    for name, label in meta.column_names_to_labels.items()
    if label and any(k in label.lower() for k in
                     ["artificial intelligence", "chatbot", "chatgpt", "generative"])
}

print("Найдено:", len(ai_vars))
for name, label in ai_vars.items():
    print(f"{name}: {label}")

Найдено: 9
ST436Q16DA: This school year, how often did you do the following tasks using <digital resources> in your school lessons? - Assess the quality of information generated by Artificial Intelligence chatbots (e.g. <ChatGPT>, <Google Bard>)
ST438Q01DA: How often do you use Artificial Intelligence chatbots (e.g. <ChatGPT>) for your school work to do the following activities? - To summarize a text I had to read
ST438Q02DA: How often do you use Artificial Intelligence chatbots (e.g. <ChatGPT>) for your school work to do the following activities? - To conduct preliminary research on a new topic
ST438Q03DA: How often do you use Artificial Intelligence chatbots (e.g. <ChatGPT>) for your school work to do the following activities? - To draft texts for writing assignments
ST438Q04DA: How often do you use Artificial Intelligence chatbots (e.g. <ChatGPT>) for your school work to do the following activities? - To help me learn
IC170Q10DA: This school year, how often did you use the following

In [9]:
df_ai, _ = pyreadstat.read_sav(
    path, usecols=["CNT", "ST438Q03DA", "ST438Q04DA", "IC170Q10DA", "ESCS"]
)
print(df_ai.notna().mean().round(3))

CNT           1.000
ST438Q03DA    0.821
ST438Q04DA    0.818
IC170Q10DA    0.373
ESCS          0.901
dtype: float64


In [10]:
cov = (df_ai.groupby("CNT")[["ST438Q03DA", "IC170Q10DA", "ESCS"]]
             .apply(lambda g: g.notna().mean())
             .round(2))
cov["n"] = df_ai.groupby("CNT").size()
print(cov.sort_values("n", ascending=False).to_string())

     ST438Q03DA  IC170Q10DA  ESCS      n
CNT                                     
ARE        0.90        0.81  0.91  34639
BRA        0.80        0.00  0.91  30140
ESP        0.88        0.79  0.95  29966
CAN        0.77        0.00  0.90  24601
KAZ        0.92        0.00  0.93  22680
GBR        0.87        0.17  0.90  15546
AUS        0.93        0.89  0.95  14658
IDN        0.96        0.00  0.98  14168
QCI        1.00        0.97  1.00  12385
ARG        0.69        0.51  0.88  11092
PSE        0.78        0.00  0.85   9647
ECU        0.94        0.00  0.97   9128
ITA        0.95        0.87  0.97   8946
UZB        0.92        0.00  0.90   8652
THA        0.97        0.00  0.97   8547
CZE        0.90        0.00  0.95   8462
JOR        0.86        0.57  0.90   8290
PHL        0.97        0.00  0.97   8072
MEX        0.95        0.00  0.98   7843
ROU        0.90        0.76  0.95   7725
LTU        0.89        0.00  0.89   7718
TUR        0.97        0.90  0.97   7702
FIN        0.90 

In [11]:
decision = """# Country selection — PISA 2025

Final sample: ARE, ESP, AUS, BRA, KAZ, GBR

| Country | n | ST438 coverage | IC170 coverage | ESCS coverage |
|---|---|---|---|---|
| ARE | 34,639 | 0.90 | 0.81 | 0.91 |
| BRA | 30,140 | 0.80 | 0.00 | 0.91 |
| ESP | 29,966 | 0.88 | 0.79 | 0.95 |
| KAZ | 22,680 | 0.92 | 0.00 | 0.93 |
| GBR | 15,546 | 0.87 | 0.17 | 0.90 |
| AUS | 14,658 | 0.93 | 0.89 | 0.95 |

Rationale: six world regions (Gulf, Southern Europe, Oceania,
Latin America, Central Asia, North-Western Europe); all six have
ST438 coverage above 0.80 and ESCS above 0.90.

Main specification uses ST438 (how AI is used) and ESCS.
IC170 (school vs home use) is available only in ARE, ESP and AUS,
so it is used for robustness on a sub-sample, never in the main model.

Excluded: USA (ST438 0.51, ESCS 0.55 — coverage too low to include).
Canada dropped in favour of Australia (ST438 0.77 vs 0.93, no ICT block).
"""

with open("../docs/country_selection.md", "w") as f:
    f.write(decision)

print("Сохранено")

Сохранено


In [12]:
pv = [n for n in meta.column_names if n.startswith("PV")]
print("Всего PV-переменных:", len(pv))

domains = sorted({n[3:] for n in pv})
print("Домены:", domains)

Всего PV-переменных: 111
Домены: ['0CMOD', '0CMPS', '0CPPK', '0CPRO', '0MATH', '0READ', '0SCIE', '0SEDE', '0SEID', '0SENV', '0SEPS', 'CMOD', 'CMPS', 'CPPK', 'CPRO', 'MATH', 'READ', 'RSCUSE', 'SCIE', 'SEDE', 'SEID', 'SENV', 'SEPS']


In [13]:
df_pv, _ = pyreadstat.read_sav(
    path, usecols=["CNT", "PV1MATH", "PV1READ", "PV1SCIE", "ST438Q03DA", "ESCS"]
)

six = ["ARE", "ESP", "AUS", "BRA", "KAZ", "GBR"]
sub = df_pv[df_pv["CNT"].isin(six)]

print(sub.groupby("CNT")[["PV1MATH", "PV1READ", "PV1SCIE"]]
         .apply(lambda g: g.notna().mean()).round(3).to_string())
print("\nРазмер подвыборки:", len(sub))

     PV1MATH  PV1READ  PV1SCIE
CNT                           
ARE      1.0      1.0      1.0
AUS      1.0      1.0      1.0
BRA      1.0      1.0      1.0
ESP      1.0      1.0      1.0
GBR      1.0      1.0      1.0
KAZ      1.0      1.0      1.0

Размер подвыборки: 147629


In [14]:
for d in ["RSCUSE", "CMOD", "CPRO", "SEDE"]:
    name = f"PV1{d}"
    print(f"{name}: {meta.column_names_to_labels.get(name)}")
    

PV1RSCUSE: None
PV1CMOD: Plausible Value 1 in LDW Subscale - Modelling 
PV1CPRO: Plausible Value 1 in LDW Subscale - Programming 
PV1SEDE: Plausible Value 1 in Competency Subscale of Science - Evaluate Designs for Scientific Enquiry 


In [15]:
df_ldw, _ = pyreadstat.read_sav(
    path, usecols=["CNT", "PV1CMOD", "PV1CPRO"]
)
ldw = df_ldw[df_ldw["CNT"].isin(six)]
print(ldw.groupby("CNT")[["PV1CMOD", "PV1CPRO"]]
         .apply(lambda g: g.notna().mean()).round(3).to_string())

     PV1CMOD  PV1CPRO
CNT                  
ARE      1.0      1.0
AUS      1.0      1.0
BRA      1.0      1.0
ESP      1.0      1.0
GBR      1.0      1.0
KAZ      1.0      1.0


In [16]:
note = """# Dependent variable — options (PISA 2025)

All six countries have full coverage (1.00) for:
- PV1-10 MATH, READ, SCIE (core domains)
- PV1-10 CMOD (LDW subscale: Modelling)
- PV1-10 CPRO (LDW subscale: Programming)

Sub-sample of the six countries: 147,629 students.

LDW = Learning in the Digital World, new domain in 2025.
Measures capacity to learn with digital tools — conceptually
much closer to the research question than attainment in maths.

Open: what exactly LDW subscales measure — need the 2025
assessment framework before choosing. If LDW fits, it becomes
the main outcome, with MATH as a robustness check.
If not: MATH main (less sensitive to language-of-test issues
in ARE and KAZ), READ as robustness.

ST438 (how AI is used) is asked once about schoolwork in general,
not per subject — there is no domain-specific AI question.
"""

with open("../docs/outcome_variable.md", "w") as f:
    f.write(note)

print("Сохранено")

Сохранено


In [17]:
note = """# LDW — what it measures (PISA 2025)

OECD definition: students' capacity to engage in an iterative process of
knowledge building and problem solving using computational tools. This
capacity is demonstrated by effective self-regulated learning while
applying computational problem-solving practices.

Developed by ACER. First assessed in PISA 2025 as an innovative domain.
Main domain of the 2025 cycle was science (not mathematics).

Subscales in the data: PV1-10 CMOD (Modelling), PV1-10 CPRO (Programming).
Full coverage (1.00) in all six selected countries.

OPEN: how the two subscales are constructed, and what the framework says
LDW does NOT measure — needs the framework PDF, not yet read.

Why this matters for the question: LDW measures self-regulated learning
with digital tools, which is conceptually much closer to "does AI help
students think or think for them" than attainment in maths or reading.

Source: oecd.org/en/topics/sub-issues/learning-in-the-digital-world
"""

with open("../docs/ldw_construct.md", "w") as f:
    f.write(note)

print("Сохранено")


Сохранено


In [18]:
for v in ["ST438Q01DA", "ST438Q02DA", "ST438Q03DA", "ST438Q04DA"]:
    print(v, "—", meta.column_names_to_labels[v].split(" - ")[-1])
    print("   ", meta.variable_value_labels.get(v))
    print()

ST438Q01DA — To summarize a text I had to read
    {1.0: 'Never or almost never', 2.0: 'About once or twice a year', 3.0: 'About once or twice a month', 4.0: 'About once or twice a week', 5.0: 'Every day or almost every day', 95.0: 'valid skip (filter rule)', 96.0: 'system missing', 97.0: 'not administered/not applicable', 98.0: 'invalid', 99.0: 'no response'}

ST438Q02DA — To conduct preliminary research on a new topic
    {1.0: 'Never or almost never', 2.0: 'About once or twice a year', 3.0: 'About once or twice a month', 4.0: 'About once or twice a week', 5.0: 'Every day or almost every day', 95.0: 'valid skip (filter rule)', 96.0: 'system missing', 97.0: 'not administered/not applicable', 98.0: 'invalid', 99.0: 'no response'}

ST438Q03DA — To draft texts for writing assignments
    {1.0: 'Never or almost never', 2.0: 'About once or twice a year', 3.0: 'About once or twice a month', 4.0: 'About once or twice a week', 5.0: 'Every day or almost every day', 95.0: 'valid skip (filter ru

In [20]:
import pandas as pd

six = ["ARE", "ESP", "AUS", "BRA", "KAZ", "GBR"]

cols = (["CNT", "CNTSCHID", "CNTSTUID", "ESCS", "ST004D01T",
         "ST438Q01DA", "ST438Q02DA", "ST438Q03DA", "ST438Q04DA",
         "W_FSTUWT"]
        + [f"PV{i}{d}" for i in range(1, 11) for d in ["MATH", "READ", "SCIE", "CMOD", "CPRO"]]
        + [f"W_FSTURWT{i}" for i in range(1, 81)])

df, _ = pyreadstat.read_sav(path, usecols=cols)
df = df[df["CNT"].isin(six)].copy()

st438 = ["ST438Q01DA", "ST438Q02DA", "ST438Q03DA", "ST438Q04DA"]
df[st438] = df[st438].where(df[st438] <= 5)

print(df.shape)
print(df[st438].describe().round(2))

(147629, 140)
       ST438Q01DA  ST438Q02DA  ST438Q03DA  ST438Q04DA
count   131001.00   130375.00   129817.00   129594.00
mean         2.84        2.99        2.74        3.32
std          1.41        1.36        1.41        1.39
min          1.00        1.00        1.00        1.00
25%          1.00        2.00        1.00        2.00
50%          3.00        3.00        3.00        4.00
75%          4.00        4.00        4.00        4.00
max          5.00        5.00        5.00        5.00


In [21]:
df.to_parquet("../data/processed/pisa2025_six.parquet")
print("Сохранено:", df.shape)

ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

In [25]:
import os
os.makedirs("../data/processed", exist_ok=True)

In [26]:
df.to_parquet("../data/processed/pisa2025_six.parquet")
print("Сохранено:", df.shape)

ArrowKeyError: No type extension with name arrow.py_extension_type found

In [27]:
df.to_pickle("../data/processed/pisa2025_six.pkl")
print("Сохранено:", df.shape)


Сохранено: (147629, 140)


In [28]:
# квинтили ESCS внутри каждой страны
df["escs_q"] = (df.groupby("CNT")["ESCS"]
                  .transform(lambda s: pd.qcut(s, 5, labels=[1, 2, 3, 4, 5])))

labels = {
    "ST438Q01DA": "Summarise a text",
    "ST438Q02DA": "Preliminary research",
    "ST438Q03DA": "Draft my writing",
    "ST438Q04DA": "Help me learn",
}

tab = (df.groupby("escs_q", observed=True)[list(labels)]
         .mean()
         .rename(columns=labels)
         .round(2))

print("Средняя частота (1 = никогда, 5 = почти каждый день)\n")
print(tab.to_string())


Средняя частота (1 = никогда, 5 = почти каждый день)

        Summarise a text  Preliminary research  Draft my writing  Help me learn
escs_q                                                                         
1                   2.73                  2.82              2.67           3.12
2                   2.81                  2.94              2.73           3.24
3                   2.85                  3.01              2.75           3.34
4                   2.89                  3.07              2.78           3.41
5                   2.97                  3.18              2.81           3.53


In [29]:
print(df.groupby(["CNT", "escs_q"], observed=True)["ESCS"].mean().round(2).unstack().to_string())


escs_q     1     2     3     4     5
CNT                                 
ARE    -0.41  0.41  0.79  1.11  1.61
AUS    -0.86  0.06  0.64  1.03  1.48
BRA    -2.70 -1.85 -1.22 -0.48  0.51
ESP    -1.57 -0.55  0.14  0.69  1.18
GBR    -1.32 -0.37  0.26  0.79  1.36
KAZ    -1.61 -0.77 -0.21  0.29  0.84


In [30]:
by_country = (df.groupby(["CNT", "escs_q"], observed=True)[["ST438Q03DA", "ST438Q04DA"]]
                .mean().round(2).unstack())
print(by_country.to_string())

       ST438Q03DA                         ST438Q04DA                        
escs_q          1     2     3     4     5          1     2     3     4     5
CNT                                                                         
ARE          3.23  3.25  3.22  3.20  3.22       3.61  3.71  3.82  3.86  3.89
AUS          2.26  2.44  2.48  2.54  2.71       2.77  3.05  3.25  3.37  3.61
BRA          2.41  2.51  2.58  2.61  2.58       2.92  3.11  3.22  3.34  3.52
ESP          2.57  2.63  2.69  2.69  2.79       3.20  3.27  3.34  3.39  3.51
GBR          2.03  2.05  2.07  2.12  2.16       2.49  2.58  2.65  2.68  2.84
KAZ          2.96  2.95  2.96  3.05  3.02       3.18  3.17  3.26  3.38  3.45


In [32]:
import os
os.makedirs("../outputs", exist_ok=True)

tab.to_csv("../outputs/st438_by_escs_pooled.csv")
by_country.to_csv("../outputs/st438_by_escs_country.csv")

note = """# First descriptive finding (23 Oct 2026)

Mean frequency of AI use for schoolwork by within-country ESCS quintile,
six countries, PISA 2025, unweighted.

Pooled gradient (Q1 to Q5):
- Draft my writing:   2.67 -> 2.81  (+0.14)
- Summarise a text:   2.73 -> 2.97  (+0.24)
- Preliminary research: 2.82 -> 3.18 (+0.36)
- Help me learn:      3.12 -> 3.53  (+0.41)

Holds in all six countries: the substitutive use (drafting) is nearly flat
across the ESCS distribution, while the learning use rises steeply.
ARE drafting is completely flat (3.23 -> 3.22). AUS has the steepest
gradient on both items.

Reading: the socio-economic gap is not in who outsources the work —
it is in who uses AI to learn. Everyone substitutes at a similar rate;
advantaged students additionally use it to understand.

CAVEATS: unweighted means; ordinal scale averaged as continuous;
no controls; descriptive only, no causal claim.
"""

with open("../docs/finding_01_descriptive.md", "w") as f:
    f.write(note)

print("Сохранено")


Сохранено
